In [1]:
from acled_model import pre_process_data,train_evaluate_model, mark_conflict_events, create_regional_monthly_baseline
import pandas as pd

INFO:ingest.acled:Access token correctly retrieved.
INFO:acled_model:Data grouped by sub_event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.


In [2]:
all_data = pd.read_csv("../data/all_data.csv")

countries = ["Sudan"]
start_date = "2017-07-01" # TODO validation for 6 month warm up period
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date = "2023-12-31"

active_start_date = "2024-01-01"
active_end_date = "2024-12-31"

## Testing to see which k deviations from the mean works best

In [7]:
test_ks = [0.25, 0.5, 0.75, 1.0, 1.25, 1.5]
results = []

for k in test_ks:
    result = train_evaluate_model(all_data, k, "event_type")
    result["k"] = k
    results.append(result)

print(results)

INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.25 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.75 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.0 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.25 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 1.5 standard deviations above the mean.


[{'optimal_threshold': '0.0345', 'onset_aupr': '0.4706', 'onset_precision_class1': '0.4229', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5944', 'active_aupr': '0.4715', 'active_precision_class1': '0.4318', 'active_recall_class1': '0.9896', 'active_f1_class1': '0.6013', 'k': 0.25}, {'optimal_threshold': '0.0465', 'onset_aupr': '0.3817', 'onset_precision_class1': '0.3795', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5502', 'active_aupr': '0.4096', 'active_precision_class1': '0.4019', 'active_recall_class1': '1.0000', 'active_f1_class1': '0.5733', 'k': 0.5}, {'optimal_threshold': '0.0249', 'onset_aupr': '0.4185', 'onset_precision_class1': '0.3679', 'onset_recall_class1': '0.9873', 'onset_f1_class1': '0.5361', 'active_aupr': '0.3620', 'active_precision_class1': '0.3382', 'active_recall_class1': '0.9583', 'active_f1_class1': '0.5000', 'k': 0.75}, {'optimal_threshold': '0.0007', 'onset_aupr': '0.3584', 'onset_precision_class1': '0.3167', 'onset_recall_class1': '1.0000',

## Testing to see whether sub events or events work better
I think that sub-events might be too sparse so event_type might be better

In [5]:
result = train_evaluate_model(all_data, 0.5)
print("sub_event_type", result)

result = train_evaluate_model(all_data, 0.5, "event_type")
print("event_type", result)

INFO:acled_model:Data grouped by sub_event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.
INFO:acled_model:Data grouped by event_type
INFO:acled_model:Escalation target set at 0.5 standard deviations above the mean.


sub_event_type {'optimal_threshold': '0.0880', 'onset_aupr': '0.3847', 'onset_precision_class1': '0.3825', 'onset_recall_class1': '0.9765', 'onset_f1_class1': '0.5497', 'active_aupr': '0.3989', 'active_precision_class1': '0.3991', 'active_recall_class1': '0.9884', 'active_f1_class1': '0.5686'}
event_type {'optimal_threshold': '0.0465', 'onset_aupr': '0.3817', 'onset_precision_class1': '0.3795', 'onset_recall_class1': '1.0000', 'onset_f1_class1': '0.5502', 'active_aupr': '0.4096', 'active_precision_class1': '0.4019', 'active_recall_class1': '1.0000', 'active_f1_class1': '0.5733'}


## Testing number of cv splits

In [ ]:
results = train_evaluate_model(all_data, k=0.5, event_col="event_type", n_splits=4)
print("Four splits in cv", results)

results = train_evaluate_model(all_data, k=0.5, event_col="event_type", n_splits=5)
